<a href="https://colab.research.google.com/github/ajayjai30/Battery-Mangement-System-Lead-Acid-Batteries/blob/main/FINAL_IMPLEMENTATION_BMS_(SOH_PREDICTION).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# ==========================================
# STEP 1: SETUP & REPAIR
# ==========================================
!git clone https://github.com/ajayjai30/Battery-Mangement-System-Lead-Acid-Batteries.git
%cd Battery-Mangement-System-Lead-Acid-Batteries

!pip install -r requirements.txt
!pip install gradio

import os
import shutil
import glob

print("\n🔧 RUNNING AUTO-REPAIR...")

# 1. Fix Folder Capitalization (Model -> models)
if os.path.exists("Model") and not os.path.exists("models"):
    print("   Renaming 'Model' to 'models'...")
    os.rename("Model", "models")

# 2. Ensure Scalers are accessible
scaler_files = glob.glob("**/*scaler_*.pkl", recursive=True)
os.makedirs("scalers", exist_ok=True)

for f in scaler_files:
    dst = os.path.join("scalers", os.path.basename(f))
    if os.path.abspath(f) != os.path.abspath(dst):
        shutil.copy(f, dst)
        print(f"   Moved {os.path.basename(f)} to ./scalers/")

# 3. Verify Model Exists
if os.path.exists("models/soh_model_gpu.keras"):
    print("✅ Model Found: soh_model_gpu.keras")
elif os.path.exists("models/soh_model_robust.keras"):
    print("✅ Model Found: soh_model_robust.keras")
else:
    print("❌ WARNING: Model not found. You may need to run the training pipeline.")

print("✅ Setup Complete.")

Cloning into 'Battery-Mangement-System-Lead-Acid-Batteries'...
remote: Enumerating objects: 173, done.
remote: Counting objects: 100% (76/76), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 173 (delta 59), reused 41 (delta 38), pack-reused 97 (from 1)
Receiving objects: 100% (173/173), 10.53 MiB | 35.35 MiB/s, done.
Resolving deltas: 100% (67/67), done.
/content/Battery-Mangement-System-Lead-Acid-Batteries/Battery-Mangement-System-Lead-Acid-Batteries/Battery-Mangement-System-Lead-Acid-Batteries

🔧 RUNNING AUTO-REPAIR...
   Renaming 'Model' to 'models'...
   Moved scaler_X_soh.pkl to ./scalers/
   Moved scaler_y_soh.pkl to ./scalers/
✅ Model Found: soh_model_gpu.keras
✅ Setup Complete.


In [10]:
# ==========================================
# STEP 2: DATA EXTRACTION
# ==========================================
import zipfile
import glob
import os
import shutil

print("📦 SETTING UP DATASET...")

zip_files = glob.glob("*.zip") + glob.glob("**/*.zip", recursive=True)

if zip_files:
    target_zip = zip_files[0]
    print(f"   Extracting {target_zip}...")
    with zipfile.ZipFile(target_zip, 'r') as zip_ref:
        zip_ref.extractall(".")
    print("✅ Data Extracted.")
else:
    print("⚠️ No Zip file found. Checking for CSV...")

csv_name = "processed_bms_data.csv"
if os.path.exists(csv_name):
    print(f"✅ Dataset Ready: {csv_name}")
else:
    csvs = glob.glob(f"**/{csv_name}", recursive=True)
    if csvs:
        shutil.copy(csvs[0], csv_name)
        print(f"✅ Dataset found and moved to root: {csv_name}")
    else:
        print("❌ Dataset missing! Simulation mode will fail.")

📦 SETTING UP DATASET...
   Extracting processed_bms_data.zip...
✅ Data Extracted.
✅ Dataset Ready: processed_bms_data.csv


In [15]:
%%writefile app_gradio.py
import gradio as gr
import pandas as pd
import time
import threading
import requests
from bms_predictor import BMSPredictor
import os
import glob

# ==============================================================================
# INITIALIZATION
# ==============================================================================
try:
    if not os.path.exists("models"):
        if os.path.exists("../models"):
            os.symlink("../models", "models")
            os.symlink("../scalers", "scalers")

    if not os.path.exists("models"):
        raise FileNotFoundError("Models folder missing")

    predictor = BMSPredictor(model_dir='models', scaler_dir='scalers')
    ai_status = "✅ AI Engine Online"
except Exception as e:
    predictor = None
    ai_status = f"❌ Error: {str(e)}"

# Global State
simulation_running = False
history_soh = []
history_v = []
history_i = []

# ==============================================================================
# DATA GENERATOR (CSV or CLOUD)
# ==============================================================================
def data_loop(mode, csv_path, speed, channel_id, read_key):
    global simulation_running, history_soh

    predictor.reset_history()
    history_soh.clear()

    simulation_running = True

    # --- MODE 1: CSV REPLAY ---
    if mode == "CSV Replay":
        # Auto-find CSV
        if csv_path == "processed_bms_data.csv" and not os.path.exists(csv_path):
            found = glob.glob("**/*processed_bms_data.csv", recursive=True)
            if found: csv_path = found[0]

        if not os.path.exists(csv_path):
            yield "❌ CSV Not Found", "0%", "0V", "0A", None
            return

        df = pd.read_csv(csv_path)

        for i in range(len(df)):
            if not simulation_running: break

            row = df.iloc[i]
            v, c, t = row['Voltage_V'], row['Current_A'], row['Temperature_C']

            # For CSV, raw display voltage == prediction voltage (no scaling needed)
            yield process_prediction(v, c, t, v, f"📂 Replay Row {i}/{len(df)}")
            time.sleep(speed)

    # --- MODE 2: LIVE THINGSPEAK ---
    elif mode == "Live ThingSpeak":
        url = f"https://api.thingspeak.com/channels/{channel_id}/feeds.json?api_key={read_key}&results=20"
        last_entry_id = None

        # Keep track of last values to persist them on the UI during the "waiting" state
        last_raw_v = 0.0   # Raw voltage for display
        last_v = 0.0       # Divided voltage for prediction
        last_c = 0.0

        while simulation_running:
            try:
                r = requests.get(url, timeout=5).json()
                feeds = r.get('feeds', [])

                new_data_processed = False

                for feed in feeds:
                    if not simulation_running:
                        break

                    entry_id = feed.get('entry_id')

                    # Process only if this is a new reading
                    if last_entry_id is None or entry_id > last_entry_id:
                        v_raw = feed.get('field1')
                        c_raw = feed.get('field2')
                        t_raw = feed.get('field3')

                        if v_raw is not None and c_raw is not None and t_raw is not None:
                            last_raw_v = float(v_raw)       # Raw voltage — shown in UI
                            last_v = last_raw_v / 6.0       # Divided voltage — used for prediction
                            last_c = float(c_raw)
                            t = float(t_raw)

                            # Yield prediction using divided voltage, but display raw voltage
                            yield process_prediction(last_v, last_c, t, last_raw_v, f"📡 Live Data [ID: {entry_id}]")

                            last_entry_id = entry_id
                            new_data_processed = True
                            time.sleep(0.1)

                # If no new data was found in this cycle, update status to "Listening..."
                if not new_data_processed and last_entry_id is not None:
                    last_soh_str = f"{history_soh[-1]:.2f}%" if history_soh else "--"

                    df_plot = pd.DataFrame({"Step": range(len(history_soh)), "SOH": history_soh}) if history_soh else None

                    yield (
                        "⏳ Listening and waiting for new values...",
                        last_soh_str,
                        f"{last_raw_v:.2f} V",   # Show raw voltage in waiting state too
                        f"{last_c:.2f} A",
                        df_plot
                    )

            except Exception as e:
                yield f"⚠️ Connection Error: {str(e)[:20]}", "Err", "Err", "Err", None

            # Poll for new values every 5 seconds
            time.sleep(5)

    yield "⏹️ Stopped", f"{history_soh[-1] if history_soh else 0:.2f}%", "0V", "0A", None


def process_prediction(v, c, t, display_v, status_msg):
    """
    v         : voltage divided by 6 — used for AI prediction
    display_v : raw voltage from ThingSpeak — shown in the UI
    """
    soh = predictor.predict_realtime(v, c, t)

    if soh:
        history_soh.append(soh)
        df_plot = pd.DataFrame({"Step": range(len(history_soh)), "SOH": history_soh})
        return status_msg, f"{soh:.2f}%", f"{display_v:.2f} V", f"{c:.2f} A", df_plot
    else:
        return f"{status_msg} (Buffering...)", "--", f"{display_v:.2f} V", f"{c:.2f} A", None


def stop_simulation():
    global simulation_running
    simulation_running = False
    return "⏹️ Stopping..."

# ==============================================================================
# UI LAYOUT
# ==============================================================================
with gr.Blocks(title="BMS Cloud Monitor", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🔋 AI Battery SOH Monitor (IoT Edition)")
    gr.Markdown(f"**System Status:** {ai_status}")

    with gr.Row():
        # --- LEFT PANEL: CONTROLS ---
        with gr.Column(scale=1):
            gr.Markdown("### 🎛️ Source Settings")

            mode_input = gr.Radio(["CSV Replay", "Live ThingSpeak"], label="Data Source", value="CSV Replay")

            # CSV Options
            with gr.Group(visible=True) as csv_group:
                csv_path = gr.Textbox(value="processed_bms_data.csv", label="Dataset Path")
                speed_input = gr.Slider(0.01, 1.0, value=0.1, label="Replay Speed")

            # ThingSpeak Options
            with gr.Group(visible=False) as ts_group:
                ts_id = gr.Textbox(label="Channel ID", placeholder="e.g. 123456")
                ts_key = gr.Textbox(label="Read API Key", placeholder="e.g. ABC12345")

            # Buttons
            with gr.Row():
                start_btn = gr.Button("▶️ Start", variant="primary")
                stop_btn = gr.Button("⏹️ Stop", variant="stop")

            status_out = gr.Textbox(label="Status", interactive=False)

        # --- RIGHT PANEL: METRICS ---
        with gr.Column(scale=3):
            gr.Markdown("### 📊 Real-time Health")
            with gr.Row():
                soh_box = gr.Textbox(label="Health (SOH)", value="--", elem_id="soh_val")
                volt_box = gr.Textbox(label="Voltage", value="--")
                curr_box = gr.Textbox(label="Current", value="--")

            plot = gr.LinePlot(
                x="Step", y="SOH", title="SOH Trend", width=600, height=300, y_lim=[0, 105]
            )

    # --- INTERACTIVITY ---
    def toggle_inputs(mode):
        return {
            csv_group: gr.update(visible=(mode == "CSV Replay")),
            ts_group: gr.update(visible=(mode == "Live ThingSpeak"))
        }

    mode_input.change(toggle_inputs, inputs=mode_input, outputs=[csv_group, ts_group])

    start_btn.click(
        fn=data_loop,
        inputs=[mode_input, csv_path, speed_input, ts_id, ts_key],
        outputs=[status_out, soh_box, volt_box, curr_box, plot]
    )

    stop_btn.click(fn=stop_simulation, inputs=None, outputs=status_out)

if __name__ == "__main__":
    demo.queue().launch(share=True, inline=False)

Overwriting app_gradio.py


In [16]:
import warnings
warnings.filterwarnings("ignore")

print("LAUNCHING GRADIO DASHBOARD")
print("Look for the 'Running on public URL' link below and also check console for status messages.")

# Run the script
!python app_gradio.py


LAUNCHING GRADIO DASHBOARD
Look for the 'Running on public URL' link below and also check console for status messages.
2026-03-05 07:14:35.241416: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772694875.262923   26155 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772694875.269951   26155 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772694875.286813   26155 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772694875.286856   26155 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 